# redis: a key-value store
* every database we've used so far in this course (sqlite3 in lesson 13, and SQLAlchemy/postgres/mysql in lessons 40-43) has been a **relational** database - data lives in tables, made of rows and columns, and we talk to it with SQL
* redis is something fundamentally different: a **key-value store** - there are no tables, no columns, no schema, and no SQL at all
* the whole mental model is: its an enormous python `dict` that lives in its own separate server process on the network, shared by many different programs at once, and its blazing fast because (unlike sqlite/postgres/mysql, which are disk-first) redis keeps everything **in RAM** - "REDIS" literally stands for **RE**mote **DI**ctionary **S**erver
* you dont "query" redis for rows matching some condition like `WHERE year < 1970` - you ask for a value by its **exact key**, full stop. thats the whole trade-off: much less flexible than SQL, but insanely fast and simple for the specific things it's good at
* a redis server is ALREADY running in this sandbox on `127.0.0.1:6379` - we dont install or start anything ourselves, we just connect to it, same as connecting to any other database server over the network

## connecting to redis
* the `redis` package (redis-py) is python's standard client library for talking to a redis server - already pip-installed for us here
* `redis.Redis(host=..., port=...)` gives us a connection object, very roughly the "connection" equivalent of `sqlite3.connect()` from lesson 13 - except this one talks to a real separate server process over the network instead of opening a local file
* `decode_responses=True` is a detail worth understanding: redis itself has no concept of "text" at all, it only stores raw bytes - without this option, every string we get back would be raw `bytes` (`b"hello"`) instead of a normal python `str`. setting it to `True` tells the client "please decode everything back to text for me", which is much nicer to work with when we know we're only storing text

In [1]:
import redis    # redis-py, the official python client library for redis - already installed in this sandbox
import time     # we'll use this for TTL demos and for timing our caching example later
import threading  # we'll need this later for running a pub/sub subscriber in the background

# connect to the redis server thats already running on this machine, on the default redis port 6379
# decode_responses=True: without it, every string value coming back would be raw bytes (b"hello") instead
# of a normal python str, since redis has no built-in concept of text encodings - it only stores bytes
r = redis.Redis(host="127.0.0.1", port=6379, decode_responses=True)

# ping() is the classic "is the server actually there and responding" health check, similar in spirit
# to how we'd check any other network service is reachable before trying to use it
print("connected:", r.ping())  # should print True

# a little housekeeping note for this whole lesson: every single key we create below is prefixed with
# "l44:" (this lesson's number). redis has no tables/schemas to keep things separated, so using ":" as
# a manual namespace separator INSIDE the key name itself is a very common real-world redis convention
# (e.g. "l44:user:1", "app:session:abc123") - it keeps our demo keys easy to spot and easy to clean up
# again at the very end, without touching anything else that might be using this same shared server
print("redis server version:", r.info("server")["redis_version"])

connected: True
redis server version: 7.0.15


## basic strings
* the simplest possible redis usage: `SET key value` stores something, `GET key` reads it back - one key always maps to exactly one value
* `INCR` atomically increments a numeric value - a small but genuinely useful feature we'll explain below
* `MSET`/`MGET` let us set/get several keys in a single round-trip to the server instead of many separate ones

In [2]:
r.set("l44:greeting", "hello redis")  # SET - stores a single value under a single key, overwrites if it already existed
print(r.get("l44:greeting"))            # GET - fetch it back by its exact key name

print(r.get("l44:does_not_exist"))  # a missing key doesnt raise an error, it just returns None (like dict.get() in python)

# INCR - atomically increments a numeric value stored as a string, and returns the new value
# "atomically" is the important word here: even if a hundred different programs called INCR on the exact
# same key at the exact same instant, redis guarantees none of those increments get silently lost - the
# whole read-modify-write happens as one indivisible step inside the server. recap lesson 11: getting this
# right yourself with a plain python variable shared across threads needs an explicit threading.Lock -
# redis just gives it to us for free, and it works across entirely separate PROGRAMS too, not just threads
r.set("l44:counter:demo", 0)          # start at 0
r.incr("l44:counter:demo")             # -> 1
r.incr("l44:counter:demo")             # -> 2
r.incr("l44:counter:demo", amount=5)  # you can increment by more than 1 in one call too -> 7
print("counter is now:", r.get("l44:counter:demo"))

# mset/mget - set/get several keys in ONE network round-trip instead of many separate ones
# each individual r.set()/r.get() call is a full request-response over the network, so batching many
# of them together with mset/mget is noticeably faster once you have a lot of keys to touch at once
r.mset({"l44:fruit:1": "apple", "l44:fruit:2": "banana", "l44:fruit:3": "cherry"})
print(r.mget("l44:fruit:1", "l44:fruit:2", "l44:fruit:3"))  # returns a list, in the same order we asked for

hello redis
None
counter is now: 7
['apple', 'banana', 'cherry']


## TTL / expiry - redis's killer feature
* this is one of the biggest reasons people reach for redis at all: a key can be told to **automatically delete itself** after N seconds, with zero extra code from us
* a plain python `dict` cant do this - you'd have to store your own timestamp alongside every value and manually check+clean it up yourself, everywhere you read it
* even a relational database like sqlite/postgres doesnt give you this for free - you'd need a scheduled job that periodically deletes old rows
* `r.set(key, value, ex=seconds)` sets a value with an expiry in one call, `r.expire(key, seconds)` adds/changes a TTL on a key that already exists, and `r.ttl(key)` tells us how many seconds are left

In [3]:
# set() can take an "ex" (expire seconds) argument directly - this key will vanish by itself after 2 seconds
r.set("l44:session:temp", "some session data", ex=2)

# ttl() tells us how many seconds are left before a key auto-expires (-1 = no expiry set, -2 = key doesnt exist)
print("ttl right after setting:", r.ttl("l44:session:temp"))

time.sleep(1)  # wait a bit...
print("ttl after ~1 second:", r.ttl("l44:session:temp"))  # should have counted down by roughly 1

time.sleep(1.5)  # ...and now wait past the 2 second mark entirely
print("value after it should have expired:", r.get("l44:session:temp"))  # None - redis deleted it for us, automatically!
print("ttl of a now-gone key:", r.ttl("l44:session:temp"))  # -2 = this key doesnt exist at all anymore

# you can also add (or change) a TTL on a key that already exists, without needing to rewrite its value at all
r.set("l44:another_key", "some value")   # no expiry set yet, this key would normally live forever
r.expire("l44:another_key", 60)           # now it will auto-expire in 60 seconds from now
print("ttl on l44:another_key:", r.ttl("l44:another_key"))

# this exact feature - "expire this automatically, no cleanup code needed" - is why redis is such a
# popular choice for caches, login sessions, rate limiters and short-lived locks in real production systems

ttl right after setting: 2


ttl after ~1 second: 1


value after it should have expired: None
ttl of a now-gone key: -2
ttl on l44:another_key: 60


## lists - an ordered sequence of values
* a redis list is an ordered collection of values under one key - good for things like a simple queue or a "recent items" log
* `RPUSH`/`LPUSH` add to the right/left end ("r"ight and "l"eft), `LRANGE` reads a range back out, `LPOP`/`RPOP` remove from either end

In [4]:
# rpush appends to the RIGHT end of the list, lpush prepends to the LEFT end
r.rpush("l44:recent_logins", "alice")   # list is now: ["alice"]
r.rpush("l44:recent_logins", "bob")     # list is now: ["alice", "bob"]
r.lpush("l44:recent_logins", "carol")   # list is now: ["carol", "alice", "bob"] - carol jumped to the very front

# lrange(key, start, stop) reads a range of elements by index, similar in spirit to python list slicing
# (0 = first element), and -1 as the "stop" means "all the way to the last element"
print("whole list right now:", r.lrange("l44:recent_logins", 0, -1))

r.rpush("l44:recent_logins", "dave", "erin")  # we can push several values onto the end in one single call too
print("after pushing more:", r.lrange("l44:recent_logins", 0, -1))

print("how many logins tracked:", r.llen("l44:recent_logins"))  # llen = list length

# a common real use case: keep only the N most recent entries in a rolling "log" list
# lpop removes and returns the leftmost (oldest, in our case) item - useful for a simple FIFO queue
oldest_login = r.lpop("l44:recent_logins")
print("oldest login, now removed from the front:", oldest_login)
print("list after popping the oldest entry:", r.lrange("l44:recent_logins", 0, -1))

whole list right now: ['carol', 'alice', 'bob']
after pushing more: ['carol', 'alice', 'bob', 'dave', 'erin']
how many logins tracked: 5
oldest login, now removed from the front: carol
list after popping the oldest entry: ['alice', 'bob', 'dave', 'erin']


## hashes - fields of one object under a single key
* a redis hash is like a small nested dict living entirely under ONE key - great for representing "all the fields of one object" without needing a separate top-level key per field
* imagine the alternative: a totally separate key per field (`l44:user:1:name`, `l44:user:1:email`, `l44:user:1:age`, ...) - technically possible, but way messier to manage, and you cant fetch "the whole object" in one call
* `HSET`/`HGET`/`HGETALL` work on individual fields or the whole hash at once

In [5]:
# HSET with a "mapping" sets several fields of the same hash in one single call
r.hset("l44:user:1", mapping={"name": "Alice", "email": "alice@example.com", "age": "30"})
# note: redis stores everything internally as strings/bytes, there is no separate "integer" field type -
# so a number going in as a field value comes back out as a string too (thats why "age" prints as "30", not 30)

print("just one field:", r.hget("l44:user:1", "name"))  # fetch a single field
print("the whole hash:", r.hgetall("l44:user:1"))          # fetch every field back at once, as a python dict

r.hset("l44:user:1", "age", 31)  # update (or add) just ONE field, without touching any of the others at all
print("after updating just the age field:", r.hgetall("l44:user:1"))

print("does user 1 have a 'phone' field?", r.hexists("l44:user:1", "phone"))  # False - we never set that one
print("all field names on this hash:", r.hkeys("l44:user:1"))

just one field: Alice
the whole hash: {'name': 'Alice', 'email': 'alice@example.com', 'age': '30'}
after updating just the age field: {'name': 'Alice', 'email': 'alice@example.com', 'age': '31'}
does user 1 have a 'phone' field? False
all field names on this hash: ['name', 'email', 'age']


## sets - unique, unordered values
* recap lesson 02: a python `set` is an unordered collection of unique values with fast membership checks and set operations (union, intersection, difference)
* a redis SET is the exact same concept, just living on the server, shared by every connected client instead of just inside one python program
* `SADD`/`SMEMBERS`/`SISMEMBER` mirror python's `.add()`/reading the whole set/`in`, and `SINTER`/`SUNION`/`SDIFF` mirror python's `&`/`|`/`-` set operators

In [6]:
# sadd adds one or more members - just like python sets, duplicates are silently ignored
r.sadd("l44:tags:article1", "python", "redis", "database", "python")  # "python" appears twice here, doesnt matter
print("article1's tags:", r.smembers("l44:tags:article1"))  # order is NOT guaranteed, exactly like a python set

r.sadd("l44:tags:article2", "python", "web", "flask")

print("is 'redis' a tag on article1?", r.sismember("l44:tags:article1", "redis"))  # membership test, like python's "in"

# set operations work here exactly like on regular python sets (recap lesson 02's & | - operators)
print("tags both articles share (intersection):", r.sinter("l44:tags:article1", "l44:tags:article2"))
print("all tags across both articles (union):", r.sunion("l44:tags:article1", "l44:tags:article2"))
print("tags only article1 has (difference):", r.sdiff("l44:tags:article1", "l44:tags:article2"))

print("how many tags does article1 have:", r.scard("l44:tags:article1"))  # scard = set cardinality (size)

article1's tags: {'database', 'python', 'redis'}
is 'redis' a tag on article1? 1
tags both articles share (intersection): {'python'}
all tags across both articles (union): {'python', 'redis', 'flask', 'database', 'web'}
tags only article1 has (difference): {'database', 'redis'}
how many tags does article1 have: 3


## sorted sets - a set where every member has a score, and stays ordered by it
* a sorted set is like a set (every member is unique) but each member ALSO carries a numeric score, and redis keeps the whole thing sorted by that score automatically, at all times
* this makes it the perfect fit for a **leaderboard**: add players with their score, and redis handles the ordering for you - no separate sort step needed, ever, no matter how often scores change
* `ZADD` adds/updates a member's score, `ZREVRANGE` reads a range in highest-first order (exactly "top N"), `ZSCORE`/`ZREVRANK` look up one specific member

In [7]:
# zadd takes a dict of {member: score} - lets set up a tiny leaderboard
r.zadd("l44:leaderboard", {"alice": 150, "bob": 300, "carol": 220, "dave": 90})

# zrevrange = read a range in REVERSE score order (highest first) - exactly "top N" for a leaderboard
# withscores=True also gives us each score back alongside the name, not just the bare name
top_3 = r.zrevrange("l44:leaderboard", 0, 2, withscores=True)  # indexes 0,1,2 -> the top 3 entries
print("top 3 players:")
for rank, (name, score) in enumerate(top_3, start=1):
    print(f"  #{rank} {name} - {int(score)} points")

# zscore looks up one specific member's score directly - no need to scan the whole leaderboard for it
print("bob's score:", r.zscore("l44:leaderboard", "bob"))

# zrevrank gives a member's 0-based rank counted from the TOP (highest score = rank 0)
# (theres also plain zrank, which counts from the bottom / lowest score instead)
print("carol's rank (0 = first place):", r.zrevrank("l44:leaderboard", "carol"))

# scores can be updated at any time by just calling zadd again with the same member name - redis
# re-sorts everything automatically, we never have to sort anything ourselves
r.zadd("l44:leaderboard", {"dave": 500})  # dave just had an amazing comeback round!
print("new top 3 after dave's comeback:")
for rank, (name, score) in enumerate(r.zrevrange("l44:leaderboard", 0, 2, withscores=True), start=1):
    print(f"  #{rank} {name} - {int(score)} points")

top 3 players:
  #1 bob - 300 points
  #2 carol - 220 points
  #3 alice - 150 points
bob's score: 300.0
carol's rank (0 = first place): 1
new top 3 after dave's comeback:
  #1 dave - 500 points
  #2 bob - 300 points
  #3 carol - 220 points


## the caching pattern (redis's single most common real-world use case)
* by far the most common thing redis is used for in real production systems: **caching** the result of something slow, so we dont have to redo the slow work every single time
* the classic approach is called **cache-aside**: before doing the expensive work, check redis first. if its there (a "cache hit"), just return it - fast. if its not there (a "cache miss"), do the expensive work once, THEN store the result in redis (with a TTL!) before returning it, so the next call is fast too
* we'll simulate "expensive work" with `time.sleep(0.3)` - stand-in for a slow database query, a slow external API call, or any heavy computation - and time both calls to make the speedup obvious and concrete

In [8]:
def get_user_profile(user_id):
    """cache-aside pattern: check redis first, only fall back to the 'real' (slow) source on a miss."""
    cache_key = f"l44:profile:{user_id}"  # one cache key per user, namespaced under our lesson prefix

    cached = r.get(cache_key)  # step 1: ALWAYS ask the cache first, before doing anything expensive
    if cached is not None:
        print(f"  (cache HIT for user {user_id})")
        return cached  # step 2a: found it already, we're done - no need to touch the slow source at all

    # step 2b: cache miss - we actually have to do the expensive work this one time
    print(f"  (cache MISS for user {user_id} - doing the slow lookup...)")
    time.sleep(0.3)  # standing in for a slow database query / external api call / heavy computation
    profile = f"profile-data-for-user-{user_id}"  # this stands in for whatever we actually looked up

    # step 3: populate the cache so the NEXT call is fast - and always attach a TTL here!
    # without a TTL, stale cached data could live forever, even long after the real underlying data changed
    r.set(cache_key, profile, ex=30)  # cache this result for 30 seconds
    return profile

# first call for user 42 - guaranteed to be a cache miss, we've never looked this user up before
start = time.perf_counter()
result_1 = get_user_profile(42)
duration_1 = time.perf_counter() - start
print("first call result:", result_1, f"- took {duration_1:.3f}s")

# second call for the exact SAME user - this time it should be sitting right there in the cache already
start = time.perf_counter()
result_2 = get_user_profile(42)
duration_2 = time.perf_counter() - start
print("second call result:", result_2, f"- took {duration_2:.3f}s")

print(f"\ncaching made the second call roughly {duration_1 / duration_2:.0f}x faster")
# this exact pattern - check cache first, compute+store on miss, always with a TTL - is used absolutely
# everywhere, from small side projects up to the biggest production web apps on the planet

  (cache MISS for user 42 - doing the slow lookup...)


first call result: profile-data-for-user-42 - took 0.302s
  (cache HIT for user 42)
second call result: profile-data-for-user-42 - took 0.001s

caching made the second call roughly 458x faster


## pub/sub - broadcasting messages in real time
* a completely different use case from caching: redis can also broadcast messages to multiple subscribers, live, as they happen
* this is similar in SPIRIT to lesson 34's multiplayer game server broadcast pattern (one sender, many receivers, delivered as soon as they happen) - except here redis itself acts as the "server" doing the broadcasting, instead of us hand-rolling our own socket server for it
* important: pub/sub is "fire and forget" - if nobody happens to be subscribed at the exact moment a message is published, that message is just gone, its NOT stored anywhere like a list would be
* we run the subscriber in a background `threading.Thread` (recap lesson 11) so it can keep listening while the rest of our notebook code keeps running - and we sprinkle in small `time.sleep()` calls after publishing purely so the background thread has an actual moment to receive and record each message before we check the results

In [9]:
received_messages = []  # the background subscriber thread appends every message it receives into this list

pubsub = r.pubsub()             # a dedicated connection just for subscribing, separate from our normal "r" connection
pubsub.subscribe("l44:news")   # subscribe to one channel - we could subscribe to several channels at once too

def listen_for_messages():
    """runs in its own background thread, continuously waiting for new pub/sub messages to arrive."""
    try:
        # pubsub.listen() is a generator that BLOCKS and yields new messages as they arrive, forever, until we close it
        for message in pubsub.listen():
            # listen() also yields an initial "subscribe" confirmation message first - we only care about real data
            if message["type"] == "message":
                received_messages.append(message["data"])  # decode_responses=True already gave us a plain python str here
    except Exception:
        # calling pubsub.close() from the main thread while we're blocked here in listen() makes the
        # underlying socket read raise - that's expected during our deliberate shutdown, so we swallow it
        # instead of letting python print a scary (but harmless) traceback for something we did on purpose
        pass

# daemon=True means: if the main program were to exit, python wont wait around for this thread to finish -
# it just gets killed automatically, so a forgotten background thread can never accidentally hang our notebook
listener_thread = threading.Thread(target=listen_for_messages, daemon=True)
listener_thread.start()

time.sleep(0.2)  # give the subscriber thread a moment to actually finish subscribing before we publish anything

# publish() sends a message to every client CURRENTLY subscribed to that channel, live, right this instant
r.publish("l44:news", "breaking: redis lesson has started")
time.sleep(0.2)  # give the background thread a moment to actually receive and record it before we check

r.publish("l44:news", "update: we covered strings, ttl, lists, hashes, sets and sorted sets so far")
time.sleep(0.2)

print("messages the background subscriber received so far:", received_messages)

# we clean up the subscriber connection explicitly and deterministically, so this notebook never hangs:
pubsub.close()  # closing it makes listen() stop blocking and the for-loop above end naturally on its own -
# without this, the background thread would just sit there forever, blocked inside listen(), waiting for a
# next message that never comes
listener_thread.join(timeout=2)  # wait briefly for the thread to actually finish exiting
print("subscriber thread still alive afterwards?", listener_thread.is_alive())  # should be False now

messages the background subscriber received so far: ['breaking: redis lesson has started', 'update: we covered strings, ttl, lists, hashes, sets and sorted sets so far']
subscriber thread still alive afterwards? False


## a note on persistence
* this particular redis server was started with `--save ""`, which means it does **not** write anything to disk at all - if it restarted right now, every key we've created in this whole lesson would simply be gone
* thats a completely deliberate choice for THIS teaching sandbox: it makes redis fast and totally disposable, with nothing left behind on disk for us to worry about cleaning up
* real production redis deployments almost always DO configure persistence, in one (or both) of two ways: **RDB snapshots** (the entire dataset gets saved to a single disk file every so often) and/or an **AOF** (append-only file) log, where every write command gets appended to a file as it happens, so it can be replayed to rebuild the exact same state after a restart
* but heres the key conceptual point either way: even WITH persistence turned on, redis is still fundamentally an **in-memory-first** store - the live, working copy of your data always lives in RAM, and disk is only ever a backup/recovery mechanism for it. this is the opposite of a relational database like sqlite/postgres/mysql, which is **disk-first** - the data genuinely lives on disk, and RAM is just a cache layer on top of that

## quick comparison to other languages
* redis-py's API maps almost directly onto the actual REDIS COMMANDS themselves (`SET`, `GET`, `EXPIRE`, `ZADD`, `PUBLISH`, ...) - and those commands are IDENTICAL no matter which language's client library sends them, because every client is just sending the exact same text-based protocol to the exact same server
* this makes redis one of the most language-agnostic pieces of infrastructure in this entire course: Java has Jedis and Lettuce, Node.js has `ioredis`/`node-redis`, C# has StackExchange.Redis, Go has `go-redis`, Rust has `redis-rs` - every single one of them exposes roughly `client.set(key, value)` / `client.get(key)` / `client.zadd(...)`, because underneath they're all just talking to the same server the same way
* learning the COMMANDS here (not just the python syntax) transfers almost 1:1 to any other language you might use redis from later
* worth briefly mentioning: **memcached** is an older, simpler alternative that only does the basic key-value-plus-TTL caching job (no lists, no sets, no sorted sets, no pub/sub, no persistence options at all) - redis is generally preferred today specifically because it does everything memcached does, PLUS all the extra data structures and features we just saw

## Exercises
1. use `INCR` to build a simple visit counter for 3 different page keys (e.g. home, about, contact) - increment some of them more often than others, then print all 3 final counts
2. add 2 more players to the `l44:leaderboard` sorted set from earlier, then print the updated top 3
3. write a SECOND cache-aside function of your own choice (pick any "expensive" operation you like) and demonstrate its own cache miss vs cache hit timing difference, the same way we did for `get_user_profile`
4. publish 3 messages in a row on a pub/sub channel and confirm a background subscriber received all 3 of them, in the correct order

In [10]:
# TODO: solve exercise 1 here (INCR-based visit counters for 3 page keys)


# TODO: solve exercise 2 here (add 2 more players to the leaderboard, print updated top 3)


# TODO: solve exercise 3 here (a second cache-aside function, with its own hit/miss timing)


# TODO: solve exercise 4 here (publish 3 messages, confirm the subscriber got all 3 in order)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [11]:
# sample solution 1 - INCR-based visit counters for 3 different pages
r.incr("l44:visits:home")     # home gets visited a lot
r.incr("l44:visits:home")
r.incr("l44:visits:home")
r.incr("l44:visits:about")    # about gets visited just once
r.incr("l44:visits:contact")  # contact gets visited the most of all
r.incr("l44:visits:contact")
r.incr("l44:visits:contact")
r.incr("l44:visits:contact")
print("home visits:", r.get("l44:visits:home"))
print("about visits:", r.get("l44:visits:about"))
print("contact visits:", r.get("l44:visits:contact"))

# sample solution 2 - add 2 more players to the leaderboard, print the updated top 3
r.zadd("l44:leaderboard", {"erin": 610, "frank": 45})  # erin storms straight into first place, frank barely scores
print("\nupdated top 3 after adding erin and frank:")
for rank, (name, score) in enumerate(r.zrevrange("l44:leaderboard", 0, 2, withscores=True), start=1):
    print(f"  #{rank} {name} - {int(score)} points")

# sample solution 3 - a second cache-aside function: this one simulates a slow "weather api" lookup
def get_weather_forecast(city):
    cache_key = f"l44:weather:{city}"
    cached = r.get(cache_key)
    if cached is not None:
        print(f"  (cache HIT for {city})")
        return cached
    print(f"  (cache MISS for {city} - calling the slow 'weather api'...)")
    time.sleep(0.3)  # standing in for a slow external api call
    forecast = f"sunny, 22C in {city}"
    r.set(cache_key, forecast, ex=30)  # cache it, with a ttl, just like before
    return forecast

start = time.perf_counter()
weather_1 = get_weather_forecast("Berlin")
weather_duration_1 = time.perf_counter() - start
start = time.perf_counter()
weather_2 = get_weather_forecast("Berlin")  # same city again - should hit the cache this time
weather_duration_2 = time.perf_counter() - start
print(f"\nfirst call: {weather_1} ({weather_duration_1:.3f}s)")
print(f"second call: {weather_2} ({weather_duration_2:.3f}s)")
print(f"speedup: roughly {weather_duration_1 / weather_duration_2:.0f}x faster on the cached call")

# sample solution 4 - publish 3 messages, confirm a fresh background subscriber gets all 3, in order
ex4_received = []              # a brand new list for this exercise's own subscriber
ex4_pubsub = r.pubsub()        # our previous pubsub connection was already closed earlier, so we need a fresh one
ex4_pubsub.subscribe("l44:news")

def ex4_listen():
    try:
        for message in ex4_pubsub.listen():
            if message["type"] == "message":
                ex4_received.append(message["data"])
    except Exception:
        # same deliberate-shutdown situation as in the pub/sub section above: closing the pubsub
        # connection from the main thread while this generator is blocked raises here - expected, so we ignore it
        pass

ex4_thread = threading.Thread(target=ex4_listen, daemon=True)  # daemon=True again, so it can never hang the notebook
ex4_thread.start()
time.sleep(0.2)  # let the subscription actually settle before we publish anything

for i in range(1, 4):
    r.publish("l44:news", f"message number {i}")  # publish 3 messages, one after another, in order
    time.sleep(0.2)  # give the background thread time to receive each one before we send the next

print("\nmessages received:", ex4_received)
expected = [f"message number {i}" for i in range(1, 4)]
print("all 3 received, in the correct order?", ex4_received == expected)

ex4_pubsub.close()          # close deterministically, same reasoning as before - avoids a hanging listen() call
ex4_thread.join(timeout=2)  # wait briefly for the thread to actually finish exiting

home visits: 3
about visits: 1
contact visits: 4

updated top 3 after adding erin and frank:
  #1 erin - 610 points
  #2 dave - 500 points
  #3 bob - 300 points
  (cache MISS for Berlin - calling the slow 'weather api'...)


  (cache HIT for Berlin)

first call: sunny, 22C in Berlin (0.301s)
second call: sunny, 22C in Berlin (0.001s)
speedup: roughly 394x faster on the cached call



messages received: ['message number 1', 'message number 2', 'message number 3']
all 3 received, in the correct order? True


In [12]:
# cleaning up after ourselves: this redis server is SHARED, so we delete every key we created in this
# lesson before finishing up - we made sure to prefix everything with "l44:" from the very start exactly
# so we could find (and remove) all of it again this easily at the end
leftover_keys = r.keys("l44:*")  # keys() with a glob-style pattern finds every key matching that pattern
print("l44:* keys found before cleanup:", len(leftover_keys))

if leftover_keys:  # delete(*keys) can take many keys at once - unpacking the list with * passes them all in one call
    r.delete(*leftover_keys)

# final verification: make sure absolutely nothing with our prefix is left behind on the shared server
remaining_keys = r.keys("l44:*")
print("l44:* keys remaining after cleanup:", len(remaining_keys))
assert remaining_keys == [], "cleanup did not fully work - some l44:* keys are still there!"
print("all clean - the shared redis instance is left exactly as we found it")

l44:* keys found before cleanup: 16
l44:* keys remaining after cleanup: 0
all clean - the shared redis instance is left exactly as we found it


* congratulation you finished this lesson

## what we learned
* what a key-value store actually is, and how fundamentally different it is from every relational database we've used so far in this course (no tables, no schema, no SQL - just exact key lookups against an in-memory server)
* connecting with `redis.Redis(...)`, `decode_responses=True`, and why redis returns raw bytes without it
* basic string operations: `set`/`get`, atomic `incr` for safe counters, and `mset`/`mget` for batching multiple keys into one round-trip
* TTL / automatic expiry (`ex=`, `expire()`, `ttl()`) - a key that deletes itself after N seconds, something neither a plain python dict nor a relational database gives you for free
* redis's other core data structures: lists (`rpush`/`lpush`/`lrange`), hashes (`hset`/`hget`/`hgetall`), sets (`sadd`/`smembers`/`sinter`/`sunion`/`sdiff`, recapping lesson 02's python sets), and sorted sets (`zadd`/`zrevrange`/`zscore`/`zrevrank`) for an always-sorted leaderboard
* the cache-aside caching pattern - redis's single most common real-world use case - and how to measure the actual speedup it gives you
* pub/sub messaging with a background `threading.Thread` subscriber (recap lesson 11), and how to shut it down cleanly with `pubsub.close()` so it never hangs
* the difference between in-memory-first (redis, even with persistence enabled) and disk-first (sqlite/postgres/mysql) storage, and the two main ways real redis deployments persist to disk (RDB snapshots, AOF logs)
* that the redis commands themselves are identical across every language's client library, making this one of the most language-agnostic corners of the entire course, and how memcached compares as a simpler, caching-only alternative